# Phase 2: Data Preprocessing and Dataset Loading
## BraTS 2023 MRI Brain Tumor Segmentation

**Objective**: Build a complete preprocessing pipeline including:
1. Data loading from NIfTI files
2. Intensity normalization (Z-score)
3. Spatial resizing/cropping
4. Train/val/test dataset split
5. PyTorch DataLoader creation

**Expected Outcome**: Preprocessed datasets ready for model training

In [7]:
# Environment setup
import os
import sys
from pathlib import Path

# Add src directory to path for importing our modules
src_path = Path.cwd() / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Python: {sys.version}")
print(f"Working dir: {Path.cwd()}")
print(f"Src path: {src_path}")

# Check GPU availability
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Working dir: /content
Src path: /content/src

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8
Using device: cuda


In [8]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import json
from typing import Dict, Tuple, List
import warnings
warnings.filterwarnings('ignore')

print("Imports successful!")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {plt.matplotlib.__version__}")
print(f"NiBabel: {nib.__version__}")

Imports successful!
NumPy: 2.0.2
Matplotlib: 3.10.0
NiBabel: 5.4.2


## Step 1: Import Data Loading and Preprocessing Modules

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# Diagnostic: Check environment and paths
import sys
import os
from pathlib import Path

print("=" * 60)
print("DIAGNOSTIC: Current Environment")
print("=" * 60)

# Check current working directory
cwd = Path.cwd()
print(f"Current working directory: {cwd}")
print(f"Files in current directory:")
for item in sorted(cwd.iterdir())[:10]:
    prefix = "📁" if item.is_dir() else "📄"
    print(f"  {prefix} {item.name}")

# Try multiple approaches to find and add src path
src_candidates = [
    cwd / "src",
    cwd.parent / "src",
    Path("d:/SIC_Capstone 2026/src"),
    Path("D:/SIC_Capstone 2026/src"),
]

src_path = None
for candidate in src_candidates:
    if candidate.exists():
        src_path = str(candidate)
        print(f"\n✓ Found src at: {src_path}")
        break

if src_path is None:
    print("\n✗ Could not find src directory!")
    print(f"Searched locations:")
    for c in src_candidates:
        print(f"  - {c} (exists: {c.exists()})")
    raise FileNotFoundError("src directory not found. Check your working directory.")

# Add to path if not already there
if src_path not in sys.path:
    sys.path.insert(0, src_path)
    print(f"Added to sys.path: {src_path}")

# Verify data module exists
data_path = Path(src_path) / "data"
print(f"\nData module path: {data_path}")
print(f"Data module exists: {data_path.exists()}")

if data_path.exists():
    print(f"Files in data directory:")
    for file in sorted(data_path.glob("*.py")):
        print(f"  - {file.name}")

print("\n" + "=" * 60)
print("Attempting imports...")
print("=" * 60)

try:
    from data.data_loader import BraTSDataLoader
    from data.preprocessor import MRIPreprocessor
    from data.dataset import BraTSDataset, DataSplitter, create_dataloaders
    
    print("\n✓ All custom modules imported successfully!")
    print("  - BraTSDataLoader")
    print("  - MRIPreprocessor")
    print("  - BraTSDataset")
    print("  - DataSplitter")
    print("  - create_dataloaders")
except ImportError as e:
    print(f"\n✗ Import error: {e}")
    print(f"\nPython path:")
    for p in sys.path[:5]:
        print(f"  - {p}")
    raise

DIAGNOSTIC: Current Environment
Current working directory: /content
Files in current directory:
  📁 .config
  📁 drive
  📁 sample_data

✗ Could not find src directory!
Searched locations:
  - /content/src (exists: False)
  - /src (exists: False)
  - d:/SIC_Capstone 2026/src (exists: False)
  - D:/SIC_Capstone 2026/src (exists: False)


FileNotFoundError: src directory not found. Check your working directory.

## Step 2: Configure Dataset Paths and Parameters

In [ ]:
# Configuration
# Dataset root - resolve possible locations
POSSIBLE_ROOTS = [
    r"D:\SIC_Capstone 2026\Datasets\brats2023-gli-dataset\ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData",
    r"d:\SIC_Capstone 2026\Datasets\brats2023-gli-dataset\ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData",
    Path("/content/drive/MyDrive/BraTS2023/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"),
]

data_root = None
for path_candidate in POSSIBLE_ROOTS:
    if Path(path_candidate).exists():
        data_root = str(path_candidate)
        break

if data_root is None:
    raise FileNotFoundError("Could not find BraTS dataset. Check paths in POSSIBLE_ROOTS")

print(f"✓ Dataset root found: {data_root}")

# Preprocessing parameters
PREPROCESSING_CONFIG = {
    "target_shape": (240, 240, 160),  # Default model input size
    "normalize_method": "z_score",      # Options: "z_score", "min_max", "none"
    "device": device,
}

# Dataset split parameters
SPLIT_CONFIG = {
    "val_ratio": 0.1,
    "test_ratio": 0.1,
    "seed": 42,
}

# DataLoader parameters
DATALOADER_CONFIG = {
    "batch_size": 2,
    "num_workers": 0,
    "shuffle_train": True,
}

print("\nPreprocessing config:")
for key, val in PREPROCESSING_CONFIG.items():
    print(f"  {key}: {val}")

print("\nSplit config:")
for key, val in SPLIT_CONFIG.items():
    print(f"  {key}: {val}")

print("\nDataLoader config:")
for key, val in DATALOADER_CONFIG.items():
    print(f"  {key}: {val}")

FileNotFoundError: Could not find BraTS dataset. Check paths in POSSIBLE_ROOTS

## Step 3: Initialize Data Loader and List Cases

In [ ]:
# Initialize data loader
loader = BraTSDataLoader(data_root)

# List all cases
all_cases = loader.list_cases()
print(f"✓ Total cases in dataset: {len(all_cases)}")

# Show first 10 cases
print("\nFirst 10 cases:")
for i, case_path in enumerate(all_cases[:10]):
    case_id = case_path.name
    case_info = loader.get_case_info(case_id)
    print(f"  {i+1}. {case_id}")
    print(f"     Files: {case_info['num_files']}, Shape: {case_info.get('shape', 'N/A')}")

# Store case IDs for later
case_ids = [c.name for c in all_cases]
print(f"\n✓ Case IDs stored ({len(case_ids)} total)")

## Step 4: Test Preprocessing on Sample Case

In [ ]:
# Test on first case
sample_case_id = case_ids[0]
print(f"Testing preprocessing on: {sample_case_id}\n")

# Load raw data
raw_data = loader.load_case(sample_case_id)
print("Raw data loaded:")
for key, val in raw_data.items():
    print(f"  {key}: shape={val.shape}, dtype={val.dtype}, min={val.min():.3f}, max={val.max():.3f}")

# Stack modalities
modality_arrays = [raw_data[mod] for mod in loader.MODALITIES]
image_raw = np.stack(modality_arrays, axis=0)  # (4, H, W, D)
mask_raw = raw_data.get("seg", None)

print(f"\nStacked image shape: {image_raw.shape}")
print(f"Mask shape: {mask_raw.shape if mask_raw is not None else 'None'}")

# Apply preprocessing
preprocessor = MRIPreprocessor(
    target_shape=PREPROCESSING_CONFIG["target_shape"],
    normalize_method=PREPROCESSING_CONFIG["normalize_method"],
)

image_processed, mask_processed = preprocessor.preprocess(
    image_raw, mask_raw, normalize=True, crop_pad=True
)

print(f"\nProcessed image shape: {image_processed.shape}")
print(f"Processed image stats:")
for c in range(image_processed.shape[0]):
    print(f"  Channel {c}: mean={image_processed[c].mean():.4f}, std={image_processed[c].std():.4f}")

if mask_processed is not None:
    print(f"\nProcessed mask shape: {mask_processed.shape}")
    unique_labels = np.unique(mask_processed)
    print(f"Unique labels in mask: {unique_labels}")

## Step 5: Visualize Before and After Preprocessing

In [ ]:
# Visualize middle slice before and after preprocessing
slice_idx_raw = image_raw.shape[3] // 2
slice_idx_processed = image_processed.shape[3] // 2

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle(f"Preprocessing Comparison: {sample_case_id}", fontsize=14, fontweight='bold')

modality_names = ["T1", "T1c", "T2", "FLAIR"]

# Raw data
for c, name in enumerate(modality_names):
    ax = axes[0, c]
    raw_slice = image_raw[c, :, :, slice_idx_raw]
    ax.imshow(np.rot90(raw_slice), cmap='gray')
    ax.set_title(f"Raw {name}")
    ax.axis('off')

# Processed data
for c, name in enumerate(modality_names):
    ax = axes[1, c]
    proc_slice = image_processed[c, :, :, slice_idx_processed]
    ax.imshow(np.rot90(proc_slice), cmap='gray')
    ax.set_title(f"Processed {name}")
    ax.axis('off')

plt.tight_layout()
plt.show()

print(f"✓ Visualization complete")
print(f"  Raw slice index: {slice_idx_raw} (of {image_raw.shape[3]})")
print(f"  Processed slice index: {slice_idx_processed} (of {image_processed.shape[3]})")

## Step 6: Create Train/Val/Test Data Split

In [ ]:
# Create data splitter and generate split
splitter = DataSplitter(
    data_root,
    val_ratio=SPLIT_CONFIG["val_ratio"],
    test_ratio=SPLIT_CONFIG["test_ratio"],
)

split = splitter.get_split(seed=SPLIT_CONFIG["seed"])

print("Data split generated:")
print(f"  Train: {len(split['train'])} cases")
print(f"  Val:   {len(split['val'])} cases")
print(f"  Test:  {len(split['test'])} cases")
print(f"  Total: {sum(len(v) for v in split.values())} cases")

# Save split to file
output_dir = Path("data/splits")
output_dir.mkdir(parents=True, exist_ok=True)
split_file = output_dir / "data_split.json"
splitter.save_split(split, str(split_file))

print(f"\n✓ Split saved to: {split_file}")

# Show sample cases from each split
print("\nSample cases from each split:")
for split_name in ["train", "val", "test"]:
    print(f"\n{split_name.upper()}:")
    for case_id in split[split_name][:3]:
        print(f"  - {case_id}")

## Step 7: Create PyTorch DataLoaders

In [ ]:
# Create preprocessor for dataloaders
preprocessor = MRIPreprocessor(
    target_shape=PREPROCESSING_CONFIG["target_shape"],
    normalize_method=PREPROCESSING_CONFIG["normalize_method"],
)

# Create dataloaders for each split
dataloaders = create_dataloaders(
    data_root=data_root,
    split_config=split,
    batch_size=DATALOADER_CONFIG["batch_size"],
    num_workers=DATALOADER_CONFIG["num_workers"],
    device=device,
    preprocessor=preprocessor,
)

print("DataLoaders created:")
for loader_name, loader in dataloaders.items():
    num_batches = len(loader)
    print(f"  {loader_name}: {num_batches} batches")

# Save dataloader config
config = {
    "preprocessing": PREPROCESSING_CONFIG,
    "split": SPLIT_CONFIG,
    "dataloader": DATALOADER_CONFIG,
    "data_root": data_root,
}

config_file = output_dir / "config.json"
with open(config_file, "w") as f:
    json.dump(config, f, indent=2, default=str)

print(f"\n✓ Config saved to: {config_file}")

## Step 8: Test DataLoader with Sample Batch

In [ ]:
# Get a sample batch from train loader
train_loader = dataloaders["train_loader"]
batch_images, batch_masks = next(iter(train_loader))

print(f"Sample batch from train loader:")
print(f"  Image batch shape: {batch_images.shape}")
print(f"  Image batch dtype: {batch_images.dtype}")
print(f"  Image batch device: {batch_images.device}")
print(f"  Image batch stats: min={batch_images.min():.4f}, max={batch_images.max():.4f}, mean={batch_images.mean():.4f}")

if batch_masks is not None:
    print(f"\n  Mask batch shape: {batch_masks.shape}")
    print(f"  Mask batch dtype: {batch_masks.dtype}")
    print(f"  Mask unique values: {torch.unique(batch_masks).tolist()}")

# Visualize first sample in batch
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
fig.suptitle("Sample Batch: First Image, Middle Slice", fontsize=12, fontweight='bold')

mid_slice = batch_images.shape[4] // 2  # Middle z-slice

for c in range(4):
    ax = axes[c]
    slice_data = batch_images[0, c, :, :, mid_slice].cpu().numpy()
    ax.imshow(np.rot90(slice_data), cmap='gray')
    ax.set_title(f"Channel {c} (T{['1', '1c', '2', 'FLAIR'][c]})")
    ax.axis('off')

plt.tight_layout()
plt.show()

print(f"\n✓ DataLoader test successful!")

## Summary: Phase 2 Preprocessing Pipeline ✓

**Completed tasks:**
1. ✓ Data loader implementation for reading NIfTI files
2. ✓ Intensity normalization (Z-score)
3. ✓ Spatial resize/crop to target shape (240, 240, 160)
4. ✓ PyTorch Dataset class for loading preprocessed data
5. ✓ Train/val/test data split (80/10/10)
6. ✓ DataLoader creation with batch processing
7. ✓ Configuration and metadata saving

**Output files created:**
- `src/data/data_loader.py` - BraTSDataLoader class
- `src/data/preprocessor.py` - MRIPreprocessor class  
- `src/data/dataset.py` - BraTSDataset and DataSplitter classes
- `data/splits/data_split.json` - Train/val/test case splits
- `data/splits/config.json` - Preprocessing configuration

**Next steps (Phase 3):**
- Implement 3D U-Net model architecture
- Implement Swin UNETR model architecture
- Set up training loop with optimizer, loss function, metrics
- Create evaluation metrics (Dice, IoU, HD95, Precision, Recall)